In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mping
import random

import os

# Visualisando e manipulando dados

In [2]:
CAMINHO = "C:\\Users\\Nitro\\Desktop\\Deep_Learning\\ID para o jhonny-20260515T045215Z-3-001\\ID para o jhonny"

In [3]:
diretorios = os.listdir(CAMINHO)

# Dicionário chave: diretório, valor: lista de imagens
arquivos = {dir : os.listdir(CAMINHO + "\\" + dir) for dir in diretorios if dir[-4] != "."}
#print(arquivos[list(arquivos.keys())[0]])
# Lista de familias para criar datasets no estilo do tensorflow
lista_familias = list(arquivos.keys())
#print(lista_familias)
arquivos = {dir: arquivos[dir] for dir in arquivos.keys() if len(arquivos[dir]) > 20}
arquivos.keys()




dict_keys(['Arecaceae', 'Bignoniaceae', 'Euphorbiaceae', 'Fabaceae', 'Lauraceae', 'Meliaceae', 'Myrtaceae'])

# Verificando quantidade de imagens por classe

# Criando caminhos no estilo do tensor flow
# FEITO

In [4]:
caminho_test ="C:\\Users\\Nitro\\PyCharmMiscProject\\Projetos_DeepLearning\\Identificador_Plantas_Brasil\\plantas_dataset\\test"
caminho_treino = "C:\\Users\\Nitro\\PyCharmMiscProject\\Projetos_DeepLearning\\Identificador_Plantas_Brasil\\plantas_dataset\\train"
caminho_validation = "C:\\Users\\Nitro\\PyCharmMiscProject\\Projetos_DeepLearning\\Identificador_Plantas_Brasil\\plantas_dataset\\val"

In [5]:
def construir_caminhos(test, train, val):
    for familia in lista_familias:
        os.makedirs(test + "\\" + familia, exist_ok=True)
        os.makedirs(train + "\\" + familia, exist_ok=True)
        os.makedirs(val + "\\" + familia, exist_ok=True)
        print("Pastas de famílias criadas com sucesso!!!")
#construir_caminhos(caminho_test, caminho_treino, caminho_validation)
# Pastas de famílias foram criadas com sucesso

# Criando função que copia fotos
# FEITO

In [6]:
def copy_image(caminho_origem: str, percents, caminho_destino_test: str, caminho_destino_train:str, caminho_destino_val:str):
    import random
    import shutil



    # Variável "arquivos"
    for familia in arquivos.keys():
        print(familia)
        imagens_treino = random.sample(arquivos[familia], int(len(arquivos[familia]) * percents["train"]))
        sobra_treino = [x for x in arquivos[familia] if x not in imagens_treino]
        imagens_teste = random.sample(sobra_treino, int(len(sobra_treino) * 0.66))
        imagens_val = [x for x in sobra_treino if x not in imagens_teste]

        for foto in imagens_treino:
            caminho_completo_origem = CAMINHO + "\\" + familia + "\\" + foto
            caminho_completo_destino_treino = caminho_destino_train + "\\" + familia
            shutil.copy2(caminho_completo_origem, caminho_completo_destino_treino)


        for foto in imagens_teste:
            caminho_completo_origem = CAMINHO + "\\" + familia + "\\" + foto
            caminho_completo_destino_teste = caminho_destino_test + "\\" + familia

            shutil.copy2(caminho_completo_origem, caminho_completo_destino_teste)

        for foto in imagens_val:
            caminho_completo_origem = CAMINHO + "\\" + familia + "\\" + foto
            caminho_completo_destino_val = caminho_destino_val + "\\" + familia

            shutil.copy2(caminho_completo_origem, caminho_completo_destino_val)




    return
#copy_image(CAMINHO, {"test": 0.2, "train": 0.7, "val": 0.1}, caminho_test, caminho_treino, caminho_validation)

# Carregar as imagens para a rede neural

In [7]:
import tensorflow as tf
print(tf.__version__)

2.21.0


In [8]:
from tensorflow.keras.utils import image_dataset_from_directory

train_dataset = image_dataset_from_directory(caminho_treino, image_size=(192, 192), batch_size=32)
val_dataset = image_dataset_from_directory(caminho_validation, image_size=(192, 192), batch_size=32)
test_dataset = image_dataset_from_directory(caminho_test, image_size=(192,192), batch_size=32)

Found 190 files belonging to 7 classes.
Found 33 files belonging to 7 classes.
Found 51 files belonging to 7 classes.


In [9]:
for data_batch, labels_batch in train_dataset:
    print("data batch shape:", data_batch.shape)
    print("labels batch shape:", labels_batch.shape)
    print(data_batch[0].shape)
    break

data batch shape: (32, 192, 192, 3)
labels batch shape: (32,)
(192, 192, 3)


# Treinando o Modelo

In [19]:
from tensorflow import keras
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.layers import RandomFlip, RandomRotation, RandomZoom, RandomContrast
from tensorflow.keras.layers import Rescaling
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

base_model = MobileNetV2(input_shape=(192, 192, 3),
                         include_top=False,
                         weights="imagenet")

base_model.trainable = False

model = keras.Sequential([
    keras.Input(shape=(192, 192, 3)),

    #Rescaling(1./255),
    RandomFlip('horizontal'),
    RandomRotation(0.1),
    RandomZoom(0.2),
    RandomContrast(0.1),

    keras.layers.Lambda(preprocess_input),

    base_model,

    tf.keras.layers.GlobalAveragePooling2D(),
    Dense(128, activation='relu'),
    Dropout(0.4),
    Dense(7, activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

In [20]:
history = model.fit(train_dataset, epochs=20)

Epoch 1/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 8s 494ms/step - accuracy: 0.2789 - loss: 1.9335
Epoch 2/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 4s 530ms/step - accuracy: 0.4421 - loss: 1.4537
Epoch 3/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 16s 2s/step - accuracy: 0.5316 - loss: 1.2901
Epoch 4/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 23s 3s/step - accuracy: 0.5895 - loss: 1.0646
Epoch 5/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 19s 3s/step - accuracy: 0.5947 - loss: 1.0308
Epoch 6/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 19s 3s/step - accuracy: 0.6737 - loss: 0.9151
Epoch 7/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 19s 3s/step - accuracy: 0.6316 - loss: 0.9332
Epoch 8/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 19s 3s/step - accuracy: 0.7579 - loss: 0.7249
Epoch 9/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 18s 3s/step - accuracy: 0.8053 - loss: 0.6550
Epoch 10/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 11s 860ms/step - accuracy: 0.7632 - loss: 0.7177
Epoch 11/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 18s 2s/step - accuracy: 0.7526 - loss: 0.6840
Epoch 12/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 19s 3s/step - accuracy: 0.7842 - loss: 0.6463
Epoch 

# Resultados

In [21]:
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ random_flip_2 (RandomFlip)      │ (None, 192, 192, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_rotation_2               │ (None, 192, 192, 3)    │             0 │
│ (RandomRotation)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_zoom_2 (RandomZoom)      │ (None, 192, 192, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_contrast_1               │ (None, 192, 192, 3)    │             0 │
│ (RandomContrast)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda_2 (Lambda)               │ (None, 192, 192, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_192            │ (None, 6, 6, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_2      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,752,599 (10.50 MB)

 Trainable params: 164,871 (644.03 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

 Optimizer params: 329,744 (1.26 MB)

In [23]:
for imagens,labels in train_dataset.take(10):
    loss, acc = model.evaluate(imagens, labels, verbose=0)
    print(f"Acurácia no batch: {acc:.4f}")


Acurácia no batch: 0.8125
Acurácia no batch: 0.9375
Acurácia no batch: 0.9375
Acurácia no batch: 0.8438
Acurácia no batch: 0.9375
Acurácia no batch: 0.9667


In [22]:
train_loss, train_acc = model.evaluate(train_dataset)
val_loss, val_acc = model.evaluate(val_dataset)
test_loss, test_acc = model.evaluate(test_dataset)

print(f"Treino: {train_acc:.2%}")
print(f"Validação: {val_acc:.2%}")
print(f"Teste: {test_acc:.2%}")

6/6 ━━━━━━━━━━━━━━━━━━━━ 5s 504ms/step - accuracy: 0.9053 - loss: 0.3059
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 42ms/step - accuracy: 0.7273 - loss: 0.7629 
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 265ms/step - accuracy: 0.7451 - loss: 0.7059
Treino: 90.53%
Validação: 72.73%
Teste: 74.51%


In [16]:
num_epochs = 100